In [1]:
from ultralytics import YOLO
from pathlib import Path
import torch
import os

In [2]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CUDA GPU not detected. Training will use CPU.")

PyTorch version: 2.14.1+cpu
CUDA available: False
CUDA GPU not detected. Training will use CPU.


In [3]:
dataset_path = Path("dataset")

folders = [
    "train/images",
    "train/labels",
    "valid/images",
    "valid/labels",
    "test/images",
    "test/labels"
]

for folder in folders:
    path = dataset_path / folder
    print(f"{folder}: {path.exists()}")

train/images: True
train/labels: True
valid/images: True
valid/labels: True
test/images: True
test/labels: True


In [4]:
image_extensions = {".jpg", ".jpeg", ".png", ".webp"}

def count_images(folder):
    return len([
        file for file in folder.iterdir()
        if file.suffix.lower() in image_extensions
    ])

train_count = count_images(dataset_path / "train/images")
valid_count = count_images(dataset_path / "valid/images")
test_count = count_images(dataset_path / "test/images")

print("Training images:", train_count)
print("Validation images:", valid_count)
print("Test images:", test_count)
print("Total images:", train_count + valid_count + test_count)

Training images: 551
Validation images: 157
Test images: 80
Total images: 788


In [5]:
def count_labels(folder):
    return len(list(folder.glob("*.txt")))

train_labels = count_labels(dataset_path / "train/labels")
valid_labels = count_labels(dataset_path / "valid/labels")
test_labels = count_labels(dataset_path / "test/labels")

print("TRAIN")
print("Images:", train_count)
print("Labels:", train_labels)

print("\nVALIDATION")
print("Images:", valid_count)
print("Labels:", valid_labels)

print("\nTEST")
print("Images:", test_count)
print("Labels:", test_labels)

TRAIN
Images: 551
Labels: 551

VALIDATION
Images: 157
Labels: 157

TEST
Images: 80
Labels: 80


In [6]:
model = YOLO("yolo11n.pt")

In [7]:
results = model.train(
    data="dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=4,
    device="cpu",
    workers=2,
    project="runs",
    name="fruit_detector"
)

Ultralytics 8.4.171  Python-3.14.6 torch-2.14.1+cpu CPU (AMD Ryzen 3 3250U with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=fruit_detector, nbs=64, nms=None, opset=None, optimize=Fa

Matplotlib is building the font cache; this may take a moment.


Overriding model.yaml nc=80 with nc=3

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultralytics.nn.modules.block.C3k2            [128, 128, 1, True]           
  7                  -1  1    295424  ultralytics

In [13]:
from pathlib import Path

best_files = list(Path("runs").rglob("best.pt"))

for file in best_files:
    print(file)

runs\detect\runs\fruit_detector\weights\best.pt


In [14]:
if not best_files:
    raise FileNotFoundError("No best.pt file found inside the runs folder.")

best_model_path = best_files[-1]

print("Loading:", best_model_path)

best_model = YOLO(str(best_model_path))

print("Best trained model loaded successfully.")

Loading: runs\detect\runs\fruit_detector\weights\best.pt
Best trained model loaded successfully.


In [15]:
val_metrics = best_model.val(
    data="dataset/data.yaml",
    split="val"
)

Ultralytics 8.4.171  Python-3.14.6 torch-2.14.1+cpu CPU (AMD Ryzen 3 3250U with Radeon Graphics)
YOLO11n summary (fused): 100 layers, 2,582,737 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 57.365.5 MB/s, size: 17.6 KB)
val: Scanning C:\xampp\htdocs\YOLO-fruit-recognition\training\dataset\valid\labels.cache... 157 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 157/157 27.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 5.2s/it 51.8s.1ss
                   all        157        629      0.765      0.714      0.757      0.432
                 apple         44        190      0.868      0.936      0.961      0.608
                banana         61        165        0.6      0.581      0.571      0.329
                orange         51        274      0.826      0.624      0.739      0.359
Speed: 6.1ms preprocess, 287.0ms inference, 0.0ms loss, 1.9ms postprocess per image

In [16]:
test_metrics = best_model.val(
    data="dataset/data.yaml",
    split="test"
)

Ultralytics 8.4.171  Python-3.14.6 torch-2.14.1+cpu CPU (AMD Ryzen 3 3250U with Radeon Graphics)
YOLO11n summary (fused): 100 layers, 2,582,737 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 21.924.8 MB/s, size: 7.2 KB)
val: Scanning C:\xampp\htdocs\YOLO-fruit-recognition\training\dataset\test\labels... 80 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 80/80 310.7it/s 0.3s0.1s
val: New cache created: C:\xampp\htdocs\YOLO-fruit-recognition\training\dataset\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 5.6s/it 27.9s6.9ss
                   all         80        327      0.761      0.661      0.763      0.428
                 apple         22         80      0.853       0.75      0.826      0.554
                banana         29         60      0.587      0.633      0.681      0.357
                orange         29        187      0.842      0.598      0.782     

In [17]:
print("Test mAP50:", test_metrics.box.map50)
print("Test mAP50-95:", test_metrics.box.map)
print("Test Precision:", test_metrics.box.mp)
print("Test Recall:", test_metrics.box.mr)

Test mAP50: 0.76307450768774
Test mAP50-95: 0.4275539823988819
Test Precision: 0.7606653816159025
Test Recall: 0.6605716201844354


In [18]:
test_results = best_model.predict(
    source="dataset/test/images",
    conf=0.50,
    save=True,
    project="predictions",
    name="test_results"
)


image 1/80 C:\xampp\htdocs\YOLO-fruit-recognition\training\dataset\test\images\0_jpg.rf.120eeaa2a87e9cac636c7863f521249a.jpg: 608x640 1 apple, 293.4ms
image 2/80 C:\xampp\htdocs\YOLO-fruit-recognition\training\dataset\test\images\0_jpg.rf.e10b361db2e60abe3c8e8c73cbcf2b9c.jpg: 480x640 1 banana, 273.0ms
image 3/80 C:\xampp\htdocs\YOLO-fruit-recognition\training\dataset\test\images\101_jpg.rf.01e145dd261171869a3cc91f70cdcb0e.jpg: 608x640 1 apple, 212.2ms
image 4/80 C:\xampp\htdocs\YOLO-fruit-recognition\training\dataset\test\images\105_jpg.rf.3510a9cf95219e61feb0bde400730347.jpg: 448x640 23 oranges, 192.8ms
image 5/80 C:\xampp\htdocs\YOLO-fruit-recognition\training\dataset\test\images\107_jpg.rf.20b9b272b1fc0a371a1c26a42cfa228f.jpg: 544x640 4 oranges, 255.6ms
image 6/80 C:\xampp\htdocs\YOLO-fruit-recognition\training\dataset\test\images\109_jpg.rf.f4899f79ff105ad81164df2d958c2314.jpg: 480x640 3 bananas, 195.3ms
image 7/80 C:\xampp\htdocs\YOLO-fruit-recognition\training\dataset\test\image

In [19]:
own_results = best_model.predict(
    source="test_images/my_fruits.jpg",
    conf=0.50,
    save=True,
    project="predictions",
    name="own_images"
)


image 1/1 C:\xampp\htdocs\YOLO-fruit-recognition\training\test_images\my_fruits.jpg: 384x640 6 apples, 1 banana, 1 orange, 101.8ms
Speed: 3.1ms preprocess, 101.8ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)
Results saved to C:\xampp\htdocs\YOLO-fruit-recognition\training\runs\detect\predictions\own_images
